# Prepare the Lending Club credit benchmark

Downloads the pinned source, creates the fixed 900,000-row cohort, signs the shared split, runs the leakage check, and caches all shared artifacts in Google Drive.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
EXPERIMENT_NAME='lending_club_900k'
DATASET_REVISION=None
FORCE_REBUILD_SPLIT=False
FORCE_RERUN=False
RUN_TESTS=True
# ==================================================================


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir():
    subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else:
    subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
try:
    hf_token=userdata.get('HF_TOKEN')
except Exception:
    hf_token=None
if hf_token:
    os.environ['HF_TOKEN']=hf_token
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
import os, platform, psutil
if RUN_TESTS:
    subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
print('Python:',platform.python_version())
print('CPU count:',os.cpu_count())
print('System RAM GiB:',round(psutil.virtual_memory().total/1024**3,2))
experiment_root=Path(DRIVE_OUTPUT_DIR)/EXPERIMENT_NAME
print('Experiment root:',experiment_root)


In [ ]:
from tfmbench.data import prepare_data
result=prepare_data(
    DRIVE_OUTPUT_DIR,
    EXPERIMENT_NAME,
    revision=DATASET_REVISION,
    force_rebuild_split=FORCE_REBUILD_SPLIT,
    hf_token=hf_token,
)


In [ ]:
import json
from tfmbench.config import FEATURE_COLUMNS
print('Final feature list:')
for index,name in enumerate(FEATURE_COLUMNS,1): print(f'{index:2d}. {name}')
print('Class counts:',result.manifest['cohort']['class_counts'])
print('Manifest hash:',result.manifest['manifest_hash'])
display(result.leakage_report)
print(json.dumps(result.manifest,indent=2))
print('Shared artifacts:',sorted(path.name for path in (result.experiment_root/'shared').iterdir()))
